In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp "kaggle (2).json" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c demand-forecasting-kernels-only

In [ ]:
!unzip -o demand-forecasting-kernels-only.zip -d data/

In [ ]:
import pandas as pd
df = pd.read_csv("data/train.csv",parse_dates=["date"])
df.head()

In [ ]:
print("Dataset shape:",df.shape)
print("\nMissing values:")
print(df.isna().sum())
print("\nData info:")
print(df.info())

In [ ]:
df["month"]=df["date"].dt.month
df["dow"]=df["date"].dt.dayofweek
df=df.sort_values(["store","item","date"])
df.head()

,date,store,item,sales,month,dow
0,2013-01-01,1,1,13,1,1
1,2013-01-02,1,1,11,1,2
2,2013-01-03,1,1,14,1,3
3,2013-01-04,1,1,13,1,4
4,2013-01-05,1,1,10,1,5


In [ ]:
df["roll_7"]= df.groupby(["store","item"])["sales"].transform(lambda x:x.rolling(7).mean())
df["roll_28"]= df.groupby(["store","item"])["sales"].transform(lambda x:x.rolling(28).mean())
df.head(35)


,date,store,item,sales,month,dow,roll_7,roll_28
0,2013-01-01,1,1,13,1,1,NaN,NaN
1,2013-01-02,1,1,11,1,2,NaN,NaN
2,2013-01-03,1,1,14,1,3,NaN,NaN
3,2013-01-04,1,1,13,1,4,NaN,NaN
4,2013-01-05,1,1,10,1,5,NaN,NaN
5,2013-01-06,1,1,12,1,6,NaN,NaN
6,2013-01-07,1,1,10,1,0,11.857143,NaN
7,2013-01-08,1,1,9,1,1,11.285714,NaN
8,2013-01-09,1,1,12,1,2,11.428571,NaN
9,2013-01-10,1,1,9,1,3,10.714286,NaN


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

In [ ]:
results = []
for (store,item),g in df.groupby(["store","item"]):
  g = g.dropna()
  X = g[["dow","month","roll_7","roll_28"]]
  y = g["sales"]
  Xtr,Xte,ytr,yte = train_test_split(X,y,shuffle=False,test_size=0.2)
  model = LinearRegression().fit(Xtr,ytr)
  mae = mean_absolute_error(yte,model.predict(Xte))
  results.append([store,item,g["sales"].mean(),g["sales"].std(),mae])

forecast_df=pd.DataFrame(results,columns=["store","item","avg_daily_demand","demand_std","mae"])
forecast_df.head(10)

,store,item,avg_daily_demand,demand_std,mae
0,1,1,20.110617,6.683742,3.586614
1,1,2,53.499722,14.816673,6.188585
2,1,3,33.439689,9.950020,4.747375
3,1,4,20.089494,6.580155,3.622297
4,1,5,16.730962,5.620348,2.992537
5,1,6,53.413007,14.632353,6.225186
6,1,7,53.117843,14.908005,6.091967
7,1,8,69.906059,18.526762,7.257196
8,1,9,46.818788,13.076759,5.613969
9,1,10,66.790995,18.048498,7.262466


In [ ]:
import numpy as np
np.random.seed(42)
forecast_df["lead_time_days"]=np.random.randint(3,15,size=len(forecast_df))
forecast_df.head(10)

,store,item,avg_daily_demand,demand_std,mae,lead_time_days
0,1,1,20.110617,6.683742,3.586614,9
1,1,2,53.499722,14.816673,6.188585,6
2,1,3,33.439689,9.950020,4.747375,13
3,1,4,20.089494,6.580155,3.622297,10
4,1,5,16.730962,5.620348,2.992537,7
5,1,6,53.413007,14.632353,6.225186,9
6,1,7,53.117843,14.908005,6.091967,12
7,1,8,69.906059,18.526762,7.257196,5
8,1,9,46.818788,13.076759,5.613969,9
9,1,10,66.790995,18.048498,7.262466,13


In [ ]:
z = 1.65
forecast_df["safety_stock"] = z * forecast_df["demand_std"] *np.sqrt(forecast_df["lead_time_days"])
forecast_df["reorder_point"] = (forecast_df["avg_daily_demand"])*forecast_df["lead_time_days"] + forecast_df["safety_stock"].round().astype(int)
forecast_df[["store","item","avg_daily_demand","safety_stock","reorder_point"]].head(10)

,store,item,avg_daily_demand,safety_stock,reorder_point
0,1,1,20.110617,33.084523,213.995553
1,1,2,53.499722,59.883925,380.998332
2,1,3,33.439689,59.194256,493.715953
3,1,4,20.089494,34.333657,234.894942
4,1,5,16.730962,24.535571,142.116732
5,1,6,53.413007,72.430146,552.717065
6,1,7,53.117843,85.210694,722.414119
7,1,8,69.906059,68.354712,417.530295
8,1,9,46.818788,64.729957,486.369094
9,1,10,66.790995,107.373396,975.282935


In [ ]:
np.random.seed(7)
forecast_df["unit_cost"] = np.random.uniform(50, 500, size=len(forecast_df)).round(2)
forecast_df["holding_cost_pct"] = 0.20
forecast_df["annual_holding_cost"] = (forecast_df["safety_stock"] * forecast_df["unit_cost"] * forecast_df["holding_cost_pct"]).round(2)

In [ ]:
forecast_df.to_csv("forecast_output.csv",index=False)
df.to_csv("sales_clean.csv",index=False)
print("forecast_output.csv and sales_clean.csv saved")


forecast_output.csv and sales_clean.csv saved


SQL

In [ ]:
import sqlite3
conn= sqlite3.connect("retail.db")

In [ ]:
df.to_sql("sales", conn, if_exists="replace", index=False)
forecast_df.to_sql("inventory_params", conn, if_exists="replace", index=False)
print(" sales table created AND  inventory_params table created")

In [ ]:
q1 = """
SELECT store, item, date, sales,
    AVG(sales) OVER (
        PARTITION BY store, item
        ORDER BY date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg
FROM sales
WHERE store = 1 AND item = 1
LIMIT 35
"""
result = pd.read_sql(q1, conn)
result

In [ ]:
q2 = """
SELECT store, item, date, sales, reorder_point,
    CASE WHEN rolling_sum > reorder_point THEN 1 ELSE 0 END AS stockout_risk
FROM (
    SELECT s.store, s.item, s.date, s.sales, i.reorder_point, i.lead_time_days,
        SUM(s.sales) OVER (
            PARTITION BY s.store, s.item
            ORDER BY s.date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS rolling_sum
    FROM sales s
    JOIN inventory_params i
        ON s.store = i.store AND s.item = i.item
)
"""
stockout_df = pd.read_sql(q2, conn)
stockout_df.head()

In [ ]:
q3 = """
SELECT store, item, SUM(stockout_risk) AS risk_days
FROM (
    SELECT s.store, s.item,
        CASE WHEN
            SUM(s.sales) OVER (
                PARTITION BY s.store, s.item
                ORDER BY s.date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ) > i.reorder_point
        THEN 1 ELSE 0 END AS stockout_risk
    FROM sales s
    JOIN inventory_params i ON s.store = i.store AND s.item = i.item
)
GROUP BY store, item
ORDER BY risk_days DESC
LIMIT 10
"""
top_risk = pd.read_sql(q3, conn)
top_risk

In [ ]:
q4 = """
SELECT item,
       strftime('%Y-%m', date) AS month,
       SUM(sales) AS monthly_units,
       LAG(SUM(sales)) OVER (
           PARTITION BY item
           ORDER BY strftime('%Y-%m', date)
       ) AS prev_month
FROM sales
GROUP BY item, month
"""
monthly_trend = pd.read_sql(q4, conn)
monthly_trend.head(20)

In [ ]:
stockout_df.to_csv("stockout_flags.csv", index=False)
top_risk.to_csv("top_risk_skus.csv", index=False)


In [ ]:
from google.colab import files
files.download("stockout_flags.csv")
files.download("top_risk_skus.csv")